In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Install dependencies (PyG ships VGAE and DGI; GraphMAE is reimplemented below
# following the official repo: https://github.com/THUDM/GraphMAE)
!pip install torch-geometric


In [4]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, VGAE, DeepGraphInfomax
from sklearn.cluster import KMeans

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)



def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["PYTHONHASHSEED"]       = str(seed)
    #print(f"Seed set to {seed}")

def adj_to_edge_index(adj):
    """adj: numpy array (N, N) -> edge_index (2, num_edges)"""
    src, dst = np.where(adj > 0)
    return torch.tensor(np.array([src, dst]), dtype=torch.long)

def cluster(z, k, seed=42):
    """k-means on embeddings z: (N, d) -> labels (N,)"""
    z = z.detach().cpu().numpy() if torch.is_tensor(z) else np.asarray(z)
    km = KMeans(n_clusters=k, n_init=20, random_state=seed)
    return km.fit_predict(z)


Using device: cuda


In [5]:
def labels_to_cluster_array(labels, num_clusters=6):
    labels = np.asarray(labels)
    return np.array(
        [
            np.flatnonzero(labels == cluster_id)
            for cluster_id in range(num_clusters)
        ],
        dtype=object
    )

def cluster_array_to_txt(cluster_array):
    return "\n".join(
        " ".join(map(str, cluster))
        for cluster in cluster_array
    )

def save_cluster_txt(cluster_array, output_path):
    with open(output_path, "w") as f:
        for cluster in cluster_array:
            f.write(" ".join(map(str, cluster)) + "\n")

## CA data


In [6]:
true_K = 10
# Load CA_graph.csv, remove headers, and convert to matrix
CA_adj = pd.read_csv('data/CA_data/CA_graph.csv').values

# Load CA_time_series.csv and convert to matrix
CA_data = pd.read_csv('data/CA_data/CA_time_series.csv').values

print(f'CA_adj shape: {CA_adj.shape}')
print(f'CA_data shape: {CA_data.shape}')

adj = CA_adj
X = CA_data   # raw series used as node features (Option A, richer signal)


CA_adj shape: (58, 58)
CA_data shape: (58, 168)


## VGAE + K-means

Kipf & Welling (2016), *Variational Graph Auto-Encoders*. Official code: https://github.com/tkipf/gae.
We use the faithful PyG implementation (`torch_geometric.nn.VGAE`): two-layer GCN encoder producing
per-node Gaussian mean/log-std, inner-product decoder reconstructing the adjacency, loss = edge
reconstruction + (1/N)·KL. Node features = the raw time series flattened to a static vector.
Embeddings (posterior means) are clustered with k-means.

In [7]:
# ─────────────────────────────────────────────
# VGAE (PyG) — GCN encoder + inner-product decoder
# ─────────────────────────────────────────────
class VGAEEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1     = GCNConv(in_dim, hidden_dim)
        self.conv_mu   = GCNConv(hidden_dim, out_dim)
        self.conv_logstd = GCNConv(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        return self.conv_mu(h, edge_index), self.conv_logstd(h, edge_index)


def train_vgae(X_feat, edge_index, seed=42, hidden_dim=32, out_dim=16,
               epochs=500, lr=1e-2):
    set_seed(seed)
    x  = torch.tensor(X_feat, dtype=torch.float).to(device)
    ei = edge_index.to(device)
    N  = x.shape[0]

    model = VGAE(VGAEEncoder(x.shape[1], hidden_dim, out_dim)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        z = model.encode(x, ei)
        loss = model.recon_loss(z, ei) + (1.0 / N) * model.kl_loss()
        loss.backward()
        opt.step()
        if epoch % 100 == 0:
            print(f"[VGAE] epoch {epoch:4d} | loss {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        z = model.encode(x, ei)          # posterior means at eval time
    return z


In [8]:
edge_index = adj_to_edge_index(adj)
print("edge_index shape:", edge_index.shape)

z_vgae = train_vgae(X, edge_index, seed=42, epochs=500)
vgae_cluster = cluster(z_vgae, true_K, seed=17)
print("VGAE cluster labels computed.")
print("Label counts:", np.bincount(vgae_cluster))


edge_index shape: torch.Size([2, 288])
[VGAE] epoch  100 | loss 1.2447
[VGAE] epoch  200 | loss 1.3892
[VGAE] epoch  300 | loss 1.2345
[VGAE] epoch  400 | loss 1.2802
[VGAE] epoch  500 | loss 1.1943
VGAE cluster labels computed.
Label counts: [4 7 5 7 6 6 8 6 5 4]


VGAE cluster saved successfully to 'data/CA_data/VGAE_only_cluster_results.csv'


,node_index,vgae_labels
0,0,0
1,1,0
2,2,7
3,3,5
4,4,0


## DGI + K-means

Veličković et al. (ICLR 2019), *Deep Graph Infomax*. Official code: https://github.com/PetarV-/DGI.
We use the faithful PyG implementation (`torch_geometric.nn.DeepGraphInfomax`): one-layer GCN
encoder with PReLU (as in the official transductive setup), sigmoid-mean readout for the graph
summary, bilinear discriminator, and corruption by row-shuffling the feature matrix.
Embeddings are clustered with k-means.

In [12]:
# ─────────────────────────────────────────────
# DGI (PyG) — GCN-PReLU encoder, feature-shuffle corruption
# ─────────────────────────────────────────────
class DGIEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.conv = GCNConv(in_dim, hidden_dim)
        self.act  = nn.PReLU(hidden_dim)

    def forward(self, x, edge_index):
        return self.act(self.conv(x, edge_index))


def dgi_corruption(x, edge_index):
    return x[torch.randperm(x.size(0), device=x.device)], edge_index


def dgi_readout(z, x, edge_index):
    return torch.sigmoid(z.mean(dim=0))


def train_dgi(X_feat, edge_index, seed=42, hidden_dim=64, epochs=500, lr=1e-3):
    set_seed(seed)
    x  = torch.tensor(X_feat, dtype=torch.float).to(device)
    ei = edge_index.to(device)

    model = DeepGraphInfomax(
        hidden_channels=hidden_dim,
        encoder=DGIEncoder(x.shape[1], hidden_dim),
        summary=dgi_readout,
        corruption=dgi_corruption,
    ).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        pos_z, neg_z, summary = model(x, ei)
        loss = model.loss(pos_z, neg_z, summary)
        loss.backward()
        opt.step()
        if epoch % 100 == 0:
            print(f"[DGI] epoch {epoch:4d} | loss {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        z, _, _ = model(x, ei)
    return z


In [13]:
edge_index = adj_to_edge_index(adj)

z_dgi = train_dgi(X, edge_index, seed=42, epochs=500)
dgi_cluster = cluster(z_dgi, true_K, seed=17)
print("DGI cluster labels computed.")
print("Label counts:", np.bincount(dgi_cluster))


[DGI] epoch  100 | loss 0.3157
[DGI] epoch  200 | loss 0.1599
[DGI] epoch  300 | loss 0.2012
[DGI] epoch  400 | loss 0.0904
[DGI] epoch  500 | loss 0.0444
DGI cluster labels computed.
Label counts: [6 7 4 9 4 5 9 7 1 6]


In [15]:
import pandas as pd

# Create a DataFrame for the VGAE cluster results
dgi_cluster_df = pd.DataFrame({
    'node_index': range(len(dgi_cluster)),
    'dgi_labels': dgi_cluster
})

# Save to CSV
dgi_cluster_df.to_csv('data/CA_data/DGI_only_cluster_results.csv', index=False)
print("DGI cluster saved successfully to 'data/CA_data/DGI_only_cluster_results.csv'")
display(dgi_cluster_df.head())

DGI cluster saved successfully to 'data/CA_data/DGI_only_cluster_results.csv'


,node_index,dgi_labels
0,0,9
1,1,3
2,2,2
3,3,6
4,4,2


## GraphMAE + K-means

Hou et al. (KDD 2022), *GraphMAE: Self-Supervised Masked Graph Autoencoders*.
Official code: https://github.com/THUDM/GraphMAE. Reimplemented below following the official
repo's node-level pipeline: GAT encoder and GAT decoder, uniform random masking of node
features with a learnable [MASK] token (mask rate 0.5), re-masking of encoded representations
before decoding, and the scaled cosine error (SCE, γ = 3) on the masked nodes only.
Embeddings from the trained encoder are clustered with k-means.

In [16]:
# ─────────────────────────────────────────────
# GraphMAE — masked feature reconstruction with GAT enc/dec + SCE loss
# (follows https://github.com/THUDM/GraphMAE, node-classification setup)
# ─────────────────────────────────────────────
def sce_loss(x, y, gamma=3.0):
    """Scaled cosine error, Eq. (4) of GraphMAE."""
    x = F.normalize(x, p=2, dim=-1)
    y = F.normalize(y, p=2, dim=-1)
    return ((1.0 - (x * y).sum(dim=-1)).pow_(gamma)).mean()


class GraphMAE(nn.Module):
    def __init__(self, in_dim, hidden_dim, mask_rate=0.5, heads=4, gamma=3.0):
        super().__init__()
        self.mask_rate = mask_rate
        self.gamma     = gamma
        assert hidden_dim % heads == 0

        # GAT encoder (official default for node-level tasks)
        self.encoder = GATConv(in_dim, hidden_dim // heads, heads=heads)
        # GAT decoder mapping back to input feature space
        self.decoder = GATConv(hidden_dim, in_dim, heads=1)
        # learnable [MASK] token and encoder->decoder projection
        self.enc_mask_token = nn.Parameter(torch.zeros(1, in_dim))
        self.encoder_to_decoder = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def encoding_mask_noise(self, x):
        num_nodes = x.size(0)
        perm = torch.randperm(num_nodes, device=x.device)
        num_mask = int(self.mask_rate * num_nodes)
        mask_nodes = perm[:num_mask]

        out_x = x.clone()
        out_x[mask_nodes] = 0.0
        out_x[mask_nodes] += self.enc_mask_token
        return out_x, mask_nodes

    def forward(self, x, edge_index):
        # 1) mask input features
        use_x, mask_nodes = self.encoding_mask_noise(x)
        # 2) encode
        enc_rep = F.elu(self.encoder(use_x, edge_index))
        # 3) project and RE-MASK before decoding (key GraphMAE design)
        rep = self.encoder_to_decoder(enc_rep)
        rep[mask_nodes] = 0.0
        # 4) decode and compute SCE on masked nodes only
        recon = self.decoder(rep, edge_index)
        return sce_loss(recon[mask_nodes], x[mask_nodes], self.gamma)

    @torch.no_grad()
    def embed(self, x, edge_index):
        return F.elu(self.encoder(x, edge_index))


def train_graphmae(X_feat, edge_index, seed=42, hidden_dim=64, mask_rate=0.5,
                   epochs=500, lr=1e-3, weight_decay=2e-4):
    set_seed(seed)
    x  = torch.tensor(X_feat, dtype=torch.float).to(device)
    ei = edge_index.to(device)

    model = GraphMAE(x.shape[1], hidden_dim, mask_rate=mask_rate).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        loss = model(x, ei)
        loss.backward()
        opt.step()
        if epoch % 100 == 0:
            print(f"[GraphMAE] epoch {epoch:4d} | SCE loss {loss.item():.4f}")

    model.eval()
    z = model.embed(x, ei)
    return z


In [17]:
edge_index = adj_to_edge_index(adj)

z_gmae = train_graphmae(X, edge_index, seed=42, epochs=500)
graphmae_cluster = cluster(z_gmae, true_K, seed=17)
print("GraphMAE cluster labels computed.")
print("Label counts:", np.bincount(graphmae_cluster))


[GraphMAE] epoch  100 | SCE loss 0.0015
[GraphMAE] epoch  200 | SCE loss 0.0040
[GraphMAE] epoch  300 | SCE loss 0.0006
[GraphMAE] epoch  400 | SCE loss 0.0008
[GraphMAE] epoch  500 | SCE loss 0.0005
GraphMAE cluster labels computed.
Label counts: [5 7 4 5 5 7 8 6 4 7]


In [19]:
graphmae_cluster

array([9, 6, 1, 5, 1, 5, 9, 8, 1, 6, 5, 8, 2, 6, 3, 7, 0, 4, 3, 6, 9, 6,
       5, 7, 4, 6, 7, 0, 1, 2, 1, 4, 2, 5, 7, 3, 2, 9, 5, 7, 9, 3, 9, 9,
       4, 1, 8, 0, 0, 7, 5, 4, 8, 6, 6, 3, 0, 1], dtype=int32)

In [18]:
# Create a DataFrame to organize the CA county cluster labels
CAcounty_labels_df = pd.DataFrame({
    'node_index': range(len(vgae_cluster)),
    'vgae_labels': vgae_cluster,
    'dgi_labels': dgi_cluster,
    'graphmae_labels': graphmae_cluster
})

# Save to CSV
CAcounty_labels_df.to_csv('data/CA_data/GNN_CAcounty_cluster_results.csv', index=False)
print("Labels saved")
display(CAcounty_labels_df.head())


Labels saved


,node_index,vgae_labels,dgi_labels,graphmae_labels
0,0,0,9,9
1,1,0,3,6
2,2,7,2,1
3,3,5,6,5
4,4,0,2,1


## DMoN

In [ ]:
import copy

from torch_geometric.nn import GCNConv, DMoNPooling
from torch_geometric.utils import to_dense_adj


class DMoNModel(nn.Module):
    """
    GCN encoder followed by DMoN pooling.

    DMoN directly produces an N x K soft cluster-assignment matrix.
    """
    def __init__(
        self,
        in_dim,
        num_clusters,
        hidden_dim=64,
        dropout=0.5,
    ):
        super().__init__()

        self.dropout = dropout

        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        # The list specifies the internal assignment-MLP dimensions.
        self.pool = DMoNPooling(
            channels=[hidden_dim, hidden_dim],
            k=num_clusters,
            dropout=dropout,
        )

    def forward(self, x, edge_index, dense_adj):
        h = F.selu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = F.selu(self.conv2(h, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)

        # s has shape (1, N, K).
        s, _, _, spectral_loss, ortho_loss, cluster_loss = self.pool(
            h, dense_adj
        )

        return s, spectral_loss, ortho_loss, cluster_loss


def train_dmon(
    X_feat,
    edge_index,
    num_clusters,
    seed=42,
    hidden_dim=64,
    epochs=500,
    lr=1e-3,
    weight_decay=1e-4,
    dropout=0.5,
    spectral_weight=1.0,
    ortho_weight=1.0,
    cluster_weight=1.0,
    verbose=False,
):
    """
    Train DMoN and return direct hard cluster assignments.

    Parameters
    ----------
    X_feat : array-like, shape (N, T)
        Full nodal time-series vectors.
    edge_index : LongTensor, shape (2, E)
        PyG edge representation.
    num_clusters : int
        Desired number of clusters K.

    Returns
    -------
    labels : ndarray, shape (N,)
        DMoN hard cluster assignments in {0, ..., K-1}.
    """
    set_seed(seed)

    x = torch.as_tensor(
        X_feat, dtype=torch.float32, device=device
    )
    ei = edge_index.to(device)

    num_nodes = x.size(0)

    # DMoNPooling uses a dense adjacency matrix for its modularity loss.
    # Do not add self-loops here: GCNConv handles its own self-loops.
    dense_adj = to_dense_adj(
        ei, max_num_nodes=num_nodes
    )[0].to(device=device, dtype=x.dtype)

    model = DMoNModel(
        in_dim=x.size(1),
        num_clusters=num_clusters,
        hidden_dim=hidden_dim,
        dropout=dropout,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_loss = float("inf")
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        _, spectral_loss, ortho_loss, cluster_loss = model(
            x, ei, dense_adj
        )

        loss = (
            spectral_weight * spectral_loss
            + ortho_weight * ortho_loss
            + cluster_weight * cluster_loss
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"DMoN encountered a non-finite loss at epoch {epoch}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        if loss.item() < best_loss:
            best_loss = loss.item()

            # Store the best checkpoint on CPU to avoid unnecessary GPU use.
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        if verbose and (epoch == 1 or epoch % 100 == 0):
            print(
                f"[DMoN] epoch {epoch:4d} | "
                f"total {loss.item():.5f} | "
                f"spectral {spectral_loss.item():.5f} | "
                f"ortho {ortho_loss.item():.5f} | "
                f"cluster {cluster_loss.item():.5f}"
            )

    if best_state is None:
        raise RuntimeError("DMoN training did not produce a valid checkpoint.")

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        soft_assignments, _, _, _ = model(x, ei, dense_adj)

        # Shape: (N, K)
        soft_assignments = soft_assignments.squeeze(0)

        # DMoN is a direct clustering method, so use argmax rather
        # than running k-means on the assignment probabilities.
        labels = soft_assignments.argmax(dim=-1).cpu().numpy()

    cluster_sizes = np.bincount(
        labels, minlength=num_clusters
    )

    if np.any(cluster_sizes == 0):
        print(
            "[WARNING] DMoN produced an empty cluster. "
            f"Cluster sizes: {cluster_sizes.tolist()}"
        )

    return labels

## word cluster

In [ ]:
adj_word = pd.read_csv("data/word/word_graph.csv").iloc[:, 1:].values
X_word = pd.read_csv("data/word/word_time_series.csv").iloc[:, 1:].values

print(f'adj_word shape: {adj_word.shape}')
print(f'X_word shape: {X_word.shape}')


adj_word shape: (112, 112)
X_word shape: (112, 64)


In [ ]:
# VGAE + Kmeans
true_K = 2

edge_index = adj_to_edge_index(adj_word)
print("edge_index shape:", edge_index.shape)

z = train_vgae(X_word, edge_index, seed=42, epochs=500)
word_vgae_cluster = cluster(z, true_K, seed=42)
word_vgae_cluster = word_vgae_cluster + 1
print("VGAE cluster labels computed.")


edge_index shape: torch.Size([2, 850])
[VGAE] epoch  100 | loss 1.1970
[VGAE] epoch  200 | loss 1.1562
[VGAE] epoch  300 | loss 1.1131
[VGAE] epoch  400 | loss 1.1253
[VGAE] epoch  500 | loss 1.1000
VGAE cluster labels computed.


In [ ]:
# DGI + Kmeans
true_K = 2

edge_index = adj_to_edge_index(adj_word)

z = train_dgi(X_word, edge_index, seed=42, epochs=500)
word_dgi_cluster = cluster(z, true_K, seed=42)
word_dgi_cluster = word_dgi_cluster + 1
print("DGI cluster labels computed.")


[DGI] epoch  100 | loss 0.5871
[DGI] epoch  200 | loss 0.1549
[DGI] epoch  300 | loss 0.0834
[DGI] epoch  400 | loss 0.0938
[DGI] epoch  500 | loss 0.0366
DGI cluster labels computed.


In [ ]:
# GraphMAE + Kmeans
true_K = 2

edge_index = adj_to_edge_index(adj_word)

z = train_graphmae(X_word, edge_index, seed=42, epochs=500)
word_graphmae_cluster = cluster(z, true_K, seed=42)
word_graphmae_cluster = word_graphmae_cluster + 1
print("GraphMAE cluster labels computed.")


[GraphMAE] epoch  100 | SCE loss 0.5133
[GraphMAE] epoch  200 | SCE loss 0.4035
[GraphMAE] epoch  300 | SCE loss 0.2888
[GraphMAE] epoch  400 | SCE loss 0.3672
[GraphMAE] epoch  500 | SCE loss 0.3106
GraphMAE cluster labels computed.


In [ ]:
# Create a DataFrame to organize the word cluster labels
word_labels_df = pd.DataFrame({
    'node_index': range(len(word_vgae_cluster)),
    'vgae_labels': word_vgae_cluster,
    'dgi_labels': word_dgi_cluster,
    'graphmae_labels': word_graphmae_cluster
})

# Save to CSV
word_labels_df.to_csv('data/word/GRL_word_cluster_results.csv', index=False)
print("Labels saved")
display(word_labels_df.head())


Labels saved


,node_index,vgae_labels,dgi_labels,graphmae_labels
0,0,1,2,1
1,1,1,2,1
2,2,2,1,1
3,3,1,2,1
4,4,1,2,2


# LA_Traffic data


In [12]:
# Load the standardized sensor time series and directed adjacency.
LA_adj = pd.read_csv(
    "data/LA_traffic/adj_matrix.csv",
    header=None
).to_numpy(dtype=np.float32)

LA_data = pd.read_csv(
    "data/LA_traffic/y_data.csv",
    header=None
).to_numpy(dtype=np.float32)

LA_adj.shape, LA_data.shape

((207, 207), (207, 119))

In [31]:
# VGAE

true_K = 6

edge_index = adj_to_edge_index(LA_adj)
print("edge_index shape:", edge_index.shape)

z = train_vgae(LA_data, edge_index, seed=42, epochs=500)
LA_vgae_cluster = cluster(z, true_K, seed=42)
LA_vgae_cluster = LA_vgae_cluster + 1
print("VGAE cluster labels computed.")

LA_vgae_cluster

edge_index shape: torch.Size([2, 1515])
[VGAE] epoch  100 | loss 1.2266
[VGAE] epoch  200 | loss 1.2042
[VGAE] epoch  300 | loss 1.1602
[VGAE] epoch  400 | loss 1.0750
[VGAE] epoch  500 | loss 1.0734
VGAE cluster labels computed.


array([1, 6, 4, 3, 3, 3, 3, 4, 5, 5, 2, 4, 3, 1, 5, 3, 3, 3, 1, 2, 2, 6,
       2, 6, 1, 6, 6, 4, 1, 6, 6, 2, 2, 3, 5, 1, 1, 1, 3, 2, 2, 5, 1, 5,
       1, 4, 4, 2, 6, 6, 4, 1, 5, 1, 1, 4, 6, 2, 1, 2, 5, 1, 1, 5, 6, 6,
       4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 5, 4, 4, 3, 5, 5, 2, 5, 4, 5, 5,
       5, 5, 2, 3, 4, 3, 3, 1, 2, 3, 2, 2, 5, 6, 2, 2, 5, 4, 4, 4, 4, 6,
       4, 1, 1, 2, 1, 1, 1, 1, 2, 2, 6, 6, 2, 6, 6, 1, 1, 3, 3, 6, 5, 5,
       4, 6, 6, 6, 3, 5, 3, 6, 1, 2, 1, 1, 3, 1, 5, 6, 5, 1, 5, 5, 1, 1,
       3, 3, 2, 3, 5, 3, 3, 3, 3, 3, 2, 6, 3, 1, 1, 1, 6, 4, 5, 1, 6, 2,
       2, 6, 1, 4, 5, 5, 2, 6, 1, 5, 5, 3, 3, 4, 2, 3, 3, 3, 1, 3, 3, 5,
       3, 1, 1, 1, 4, 4, 5, 2, 3], dtype=int32)

In [32]:
LA_vgae_cluster

array([1, 6, 4, 3, 3, 3, 3, 4, 5, 5, 2, 4, 3, 1, 5, 3, 3, 3, 1, 2, 2, 6,
       2, 6, 1, 6, 6, 4, 1, 6, 6, 2, 2, 3, 5, 1, 1, 1, 3, 2, 2, 5, 1, 5,
       1, 4, 4, 2, 6, 6, 4, 1, 5, 1, 1, 4, 6, 2, 1, 2, 5, 1, 1, 5, 6, 6,
       4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 5, 4, 4, 3, 5, 5, 2, 5, 4, 5, 5,
       5, 5, 2, 3, 4, 3, 3, 1, 2, 3, 2, 2, 5, 6, 2, 2, 5, 4, 4, 4, 4, 6,
       4, 1, 1, 2, 1, 1, 1, 1, 2, 2, 6, 6, 2, 6, 6, 1, 1, 3, 3, 6, 5, 5,
       4, 6, 6, 6, 3, 5, 3, 6, 1, 2, 1, 1, 3, 1, 5, 6, 5, 1, 5, 5, 1, 1,
       3, 3, 2, 3, 5, 3, 3, 3, 3, 3, 2, 6, 3, 1, 1, 1, 6, 4, 5, 1, 6, 2,
       2, 6, 1, 4, 5, 5, 2, 6, 1, 5, 5, 3, 3, 4, 2, 3, 3, 3, 1, 3, 3, 5,
       3, 1, 1, 1, 4, 4, 5, 2, 3], dtype=int32)

In [33]:
save_cluster_txt(
    labels_to_cluster_array(LA_vgae_cluster, num_clusters=6),
    "data/LA_traffic/VGAE_LAtraffic_clusters.txt"
)

In [36]:
# DGI
true_K = 6

edge_index = adj_to_edge_index(LA_adj)

z = train_dgi(LA_data, edge_index, seed=42, epochs=500)
LA_dgi_cluster = cluster(z, true_K, seed=42)
LA_dgi_cluster = LA_dgi_cluster + 1
LA_dgi_cluster

[DGI] epoch  100 | loss 0.6162
[DGI] epoch  200 | loss 0.3423
[DGI] epoch  300 | loss 0.2539
[DGI] epoch  400 | loss 0.2623
[DGI] epoch  500 | loss 0.3100


array([3, 5, 3, 2, 6, 2, 2, 3, 5, 4, 5, 1, 6, 1, 5, 2, 2, 6, 5, 6, 5, 5,
       5, 2, 3, 2, 3, 1, 5, 3, 2, 3, 3, 2, 3, 5, 3, 3, 2, 3, 3, 4, 3, 4,
       1, 1, 3, 4, 2, 2, 3, 2, 4, 3, 3, 1, 2, 3, 1, 3, 4, 4, 3, 4, 2, 2,
       3, 5, 5, 4, 5, 5, 5, 4, 3, 4, 4, 5, 3, 1, 6, 4, 4, 3, 5, 1, 4, 4,
       5, 5, 3, 2, 1, 2, 6, 1, 6, 6, 5, 3, 4, 2, 5, 5, 4, 3, 3, 3, 1, 3,
       1, 3, 3, 5, 1, 3, 3, 3, 5, 3, 2, 3, 5, 2, 2, 1, 3, 6, 6, 2, 5, 4,
       1, 2, 2, 2, 2, 4, 2, 2, 3, 4, 3, 3, 2, 3, 4, 2, 4, 3, 4, 5, 1, 3,
       6, 6, 5, 6, 4, 6, 6, 6, 6, 6, 4, 2, 2, 3, 1, 1, 2, 1, 4, 3, 2, 5,
       3, 5, 1, 1, 4, 4, 5, 5, 3, 5, 5, 6, 6, 3, 5, 6, 2, 6, 3, 6, 2, 4,
       6, 3, 5, 1, 3, 3, 4, 5, 6], dtype=int32)

In [37]:
save_cluster_txt(
    labels_to_cluster_array(LA_dgi_cluster, num_clusters=6),
    "data/LA_traffic/DGI_LAtraffic_clusters.txt"
)

In [38]:
# GraphMAE
true_K = 6

edge_index = adj_to_edge_index(LA_adj)

z = train_graphmae(LA_data, edge_index, seed=42, epochs=500)
LA_graphmae_cluster = cluster(z, true_K, seed=42)
LA_graphmae_cluster = LA_graphmae_cluster + 1
LA_graphmae_cluster

[GraphMAE] epoch  100 | SCE loss 0.0252
[GraphMAE] epoch  200 | SCE loss 0.0171
[GraphMAE] epoch  300 | SCE loss 0.0177
[GraphMAE] epoch  400 | SCE loss 0.0190
[GraphMAE] epoch  500 | SCE loss 0.0220


array([4, 4, 5, 1, 2, 1, 1, 5, 4, 6, 3, 5, 2, 4, 4, 2, 1, 2, 4, 3, 3, 4,
       3, 1, 4, 1, 3, 5, 4, 4, 1, 4, 3, 1, 4, 4, 3, 4, 1, 3, 3, 6, 4, 6,
       4, 5, 5, 6, 1, 1, 5, 1, 6, 4, 4, 5, 1, 3, 4, 4, 6, 6, 4, 6, 1, 1,
       5, 3, 3, 6, 3, 3, 3, 6, 3, 6, 6, 6, 4, 5, 2, 6, 6, 4, 4, 5, 6, 6,
       4, 4, 4, 1, 5, 1, 2, 4, 3, 2, 3, 4, 6, 1, 3, 3, 6, 5, 5, 5, 5, 4,
       5, 4, 3, 6, 4, 4, 4, 3, 3, 3, 1, 4, 4, 1, 1, 4, 4, 2, 2, 1, 6, 6,
       5, 1, 1, 1, 1, 6, 1, 1, 3, 6, 4, 4, 1, 4, 6, 1, 6, 4, 6, 4, 4, 5,
       2, 2, 4, 2, 6, 2, 2, 2, 2, 2, 6, 1, 1, 5, 4, 4, 1, 5, 6, 5, 1, 3,
       4, 4, 4, 5, 6, 6, 4, 4, 4, 4, 4, 2, 2, 5, 3, 2, 2, 2, 3, 2, 1, 6,
       2, 3, 4, 5, 5, 4, 6, 3, 2], dtype=int32)

In [39]:
save_cluster_txt(
    labels_to_cluster_array(LA_graphmae_cluster, num_clusters=6),
    "data/LA_traffic/GraphMAE_LAtraffic_clusters.txt"
)

## Simulation Study: Scenatio 2 (Grid Graph) first

In [ ]:
from sklearn.metrics import (normalized_mutual_info_score, adjusted_rand_score,
                             homogeneity_score, completeness_score)
from scipy.optimize import linear_sum_assignment

def cluster_acc(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1
    row, col = linear_sum_assignment(w.max() - w)
    return w[row, col].sum() / y_pred.size

def purity(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return sum(np.unique(y_true[y_pred == c], return_counts=True)[1].max()
               for c in np.unique(y_pred)) / y_pred.size

def metrics(y_true, y_pred):
    return (normalized_mutual_info_score(y_true, y_pred),
            adjusted_rand_score(y_true, y_pred),
            cluster_acc(y_true, y_pred),
            homogeneity_score(y_true, y_pred),
            completeness_score(y_true, y_pred),
            purity(y_true, y_pred))

In [ ]:
use_data = 's3'     # 's1' / 's2' / 's3'
num_node = 210
epochs   = 300

data = np.load(f'data new/data_{use_data}_n{num_node}.npz')
num_seq = data['adj_matrices'].shape[0]

In [ ]:
#data['adj_matrices'][49]

In [ ]:
TRAINERS = {'VGAE': train_vgae, 'DGI': train_dgi, 'GraphMAE': train_graphmae}
cols = ['NMI', 'ARI', 'ACC', 'HOM', 'COM', 'PUR']
holder = {name: np.zeros((num_seq, 6)) for name in TRAINERS}

for seq_iter in range(num_seq):
    adj    = data['adj_matrices'][seq_iter]
    labels = data['labels'][seq_iter]
    X      = data['y'][seq_iter]              # n by T, raw series as node features
    edge_index = adj_to_edge_index(adj)
    K = len(np.unique(labels))                # true number of clusters

    for name, trainer in TRAINERS.items():
        z = trainer(X, edge_index, seed=42 + seq_iter, epochs=epochs)
        pred = cluster(z, K, seed=17)
        holder[name][seq_iter] = metrics(labels, pred)

    print(f'[INFO] seq_iter {seq_iter} done')

# Table in the same format as Tables 1-3: mean%(sd)
rows = []
for name in TRAINERS:
    m, s = holder[name].mean(0) * 100, holder[name].std(0)
    rows.append([name] + [f'{m[j]:.2f}%({s[j]:.2f})' for j in range(6)])

table = pd.DataFrame(rows, columns=['METHOD'] + cols)
print(f'Scenario {use_data}, N = {num_node}, {num_seq} realizations')
display(table)

[VGAE] epoch  100 | loss 1.4277
[VGAE] epoch  200 | loss 1.4269
[VGAE] epoch  300 | loss 1.4274
[DGI] epoch  100 | loss 0.8558
[DGI] epoch  200 | loss 0.5609
[DGI] epoch  300 | loss 0.3855
[GraphMAE] epoch  100 | SCE loss 0.4685
[GraphMAE] epoch  200 | SCE loss 0.3370
[GraphMAE] epoch  300 | SCE loss 0.3211
[INFO] seq_iter 0 done
[VGAE] epoch  100 | loss 1.4161
[VGAE] epoch  200 | loss 1.4036
[VGAE] epoch  300 | loss 1.4147
[DGI] epoch  100 | loss 0.6071
[DGI] epoch  200 | loss 0.3679
[DGI] epoch  300 | loss 0.2838
[GraphMAE] epoch  100 | SCE loss 0.2923
[GraphMAE] epoch  200 | SCE loss 0.5112
[GraphMAE] epoch  300 | SCE loss 0.2375
[INFO] seq_iter 1 done
[VGAE] epoch  100 | loss 1.4225
[VGAE] epoch  200 | loss 1.4129
[VGAE] epoch  300 | loss 1.4070
[DGI] epoch  100 | loss 0.7262
[DGI] epoch  200 | loss 0.4374
[DGI] epoch  300 | loss 0.2996
[GraphMAE] epoch  100 | SCE loss 0.3356
[GraphMAE] epoch  200 | SCE loss 0.3042
[GraphMAE] epoch  300 | SCE loss 0.2981
[INFO] seq_iter 2 done
[VGA

,METHOD,NMI,ARI,ACC,HOM,COM,PUR
0,VGAE,92.17%(0.05),94.84%(0.03),98.23%(0.01),92.18%(0.05),92.16%(0.05),98.23%(0.01)
1,DGI,78.46%(0.05),82.48%(0.05),93.79%(0.02),78.41%(0.05),78.52%(0.05),93.79%(0.02)
2,GraphMAE,81.75%(0.09),83.40%(0.10),93.35%(0.05),81.46%(0.09),82.04%(0.09),93.35%(0.05)


## Simulation: add DMoN

In [ ]:
TRAINERS = {
    'VGAE': train_vgae,
    'DGI': train_dgi,
    'GraphMAE': train_graphmae,
    'DMoN': train_dmon,
}

cols = ['NMI', 'ARI', 'ACC', 'HOM', 'COM', 'PUR']

holder = {
    name: np.zeros((num_seq, len(cols)))
    for name in TRAINERS
}

for seq_iter in range(num_seq):
    adj = data['adj_matrices'][seq_iter]
    labels = data['labels'][seq_iter]
    X = data['y'][seq_iter]  # N by T; full series as node features

    edge_index = adj_to_edge_index(adj)
    K = len(np.unique(labels))

    run_seed = 42 + seq_iter

    for name, trainer in TRAINERS.items():

        if name == 'DMoN':
            # DMoN directly returns cluster labels.
            pred = trainer(
                X,
                edge_index,
                num_clusters=K,
                seed=run_seed,
                epochs=epochs,
            )

        else:
            # VGAE, DGI, and GraphMAE return node embeddings.
            z = trainer(
                X,
                edge_index,
                seed=run_seed,
                epochs=epochs,
            )

            pred = cluster(
                z,
                K,
                seed=17,
            )

        holder[name][seq_iter] = metrics(labels, pred)

    print(
        f'[INFO] seq_iter {seq_iter + 1}/{num_seq} done'
    )


# Table in the same format as the existing simulation tables:
# percentage mean followed by SD on the original proportion scale.
rows = []

for name in TRAINERS:
    mean_values = holder[name].mean(axis=0) * 100
    sd_values = holder[name].std(axis=0)

    formatted = [
        f'{mean_values[j]:.2f}%({sd_values[j]:.2f})'
        for j in range(len(cols))
    ]

    rows.append([name] + formatted)

table = pd.DataFrame(
    rows,
    columns=['METHOD'] + cols,
)

print(
    f'Scenario {use_data}, N = {num_node}, '
    f'{num_seq} realizations'
)
display(table)

[VGAE] epoch  100 | loss 1.4157
[VGAE] epoch  200 | loss 1.4139
[VGAE] epoch  300 | loss 1.4041
[VGAE] epoch  400 | loss 1.3892
[VGAE] epoch  500 | loss 1.3736
[DGI] epoch  100 | loss 0.7897
[DGI] epoch  200 | loss 0.4688
[DGI] epoch  300 | loss 0.2586
[DGI] epoch  400 | loss 0.1856
[DGI] epoch  500 | loss 0.1427
[GraphMAE] epoch  100 | SCE loss 0.5117
[GraphMAE] epoch  200 | SCE loss 0.5044
[GraphMAE] epoch  300 | SCE loss 0.3836
[GraphMAE] epoch  400 | SCE loss 0.4607
[GraphMAE] epoch  500 | SCE loss 0.4133
[WARNING] DMoN produced an empty cluster. Cluster sizes: [81, 0, 129]
[INFO] seq_iter 1/25 done
[VGAE] epoch  100 | loss 1.4239
[VGAE] epoch  200 | loss 1.4156
[VGAE] epoch  300 | loss 1.4101
[VGAE] epoch  400 | loss 1.3875
[VGAE] epoch  500 | loss 1.3852
[DGI] epoch  100 | loss 0.8448
[DGI] epoch  200 | loss 0.5374
[DGI] epoch  300 | loss 0.2720
[DGI] epoch  400 | loss 0.1645
[DGI] epoch  500 | loss 0.1731
[GraphMAE] epoch  100 | SCE loss 0.5622
[GraphMAE] epoch  200 | SCE loss 0

,METHOD,NMI,ARI,ACC,HOM,COM,PUR
0,VGAE,82.48%(0.05),87.28%(0.04),95.52%(0.01),82.40%(0.05),82.55%(0.05),95.52%(0.01)
1,DGI,54.80%(0.05),52.23%(0.09),78.48%(0.06),54.63%(0.05),54.97%(0.05),78.65%(0.06)
2,GraphMAE,69.71%(0.08),70.27%(0.11),87.14%(0.09),69.14%(0.08),70.30%(0.08),87.58%(0.07)
3,DMoN,52.05%(0.06),48.16%(0.08),71.73%(0.07),46.26%(0.08),60.86%(0.07),71.73%(0.07)


In [ ]:
# Other metrics

# from sklearn.metrics import (normalized_mutual_info_score, adjusted_rand_score,
#                              homogeneity_score, completeness_score)
# from scipy.optimize import linear_sum_assignment

# def cluster_acc(y_true, y_pred):
#     y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
#     D = max(y_pred.max(), y_true.max()) + 1
#     w = np.zeros((D, D), dtype=np.int64)
#     for i in range(y_pred.size):
#         w[y_pred[i], y_true[i]] += 1
#     row, col = linear_sum_assignment(w.max() - w)
#     return w[row, col].sum() / y_pred.size

# def purity(y_true, y_pred):
#     y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
#     total = 0
#     for c in np.unique(y_pred):
#         labels, counts = np.unique(y_true[y_pred == c], return_counts=True)
#         total += counts.max()
#     return total / y_pred.size

# def evaluate(y_true, y_pred, name=""):
#     print(f"{name:10s} NMI {normalized_mutual_info_score(y_true, y_pred):.4f} | "
#           f"ARI {adjusted_rand_score(y_true, y_pred):.4f} | "
#           f"ACC {cluster_acc(y_true, y_pred):.4f} | "
#           f"HOM {homogeneity_score(y_true, y_pred):.4f} | "
#           f"COM {completeness_score(y_true, y_pred):.4f} | "
#           f"PUR {purity(y_true, y_pred):.4f}")
